In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [11]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [12]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [13]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [14]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!',
   'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS75gW88wyk/3ooD

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WO

In [18]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'Why did the man get fired from his job at the pasta factory?\n\nHe kept making **fusilli** mistakes!',
   'extras': {'signature': 'EtoYCtcYAWkUfRMIkDUa/BxRaeK2vblLZWblYqnmQi13OeLM7PL7lRYeGkADSeaFtGeZk1E/LY89N29tFuEbrFAqn+7DI6+g/9TATpSeCKS56GnrapWZ3P1QjPYLcA9cNisXUIlGaXZEZt4BFiiMCigkp53O7wU/96fLRQDZLegH/lVXDp+XpwEMPnsT7CjnEGNggW5ZGeR44SapJ60A9xLk5YtCDCLLJmB1P4pjxIRWdTFzozu/TBY+/PlgnMWMnstAFJgWIgeRLL7qczNoZR8N1KnMPCk2liI5T0mleRyP2HtgACbdqLDmlg86pnBZqnBwCx7ija2EnHZ/0LBqST1qJqZdh7+aLEkZBqm5Qc1MkkD0jTltpUSOwM0X27EPtzEqfZf+cthF2eb1Oswqd6bHxVakLWIjzHPRWGdF6Fd2OkdwwueHK+q/158I1TDPwUL1onQE29mC03Dl6ni6QhCOCnamqFAP64MiQggRoK7Z+ymIkM10NyrMXgHqso2WHqCrXDQ/+Mga8xOpvvro1Sujf+NulcJq9LiOfyrvYgqIvDFdRJLIXq9hZsGQp2maUvrOa5TAr1bvdMRFYxgnVYuUd+hDrB/f60B4eKG3Twp1a8kIfMRlOnSHqQXe444tnNg6gTz7pOya+0D+yF/FYMpWO1GVi7wssvJNxzv5wbTHer/hyqvo9zWrV0Rosh9E996IBYIEJ4K+euse0F23t5tEGj29COwhpCxJCXofaOQieYcM6cAoVtcylMm0oC5W14zNzxyRMs9cmmObEVHDNrtnwR3Dzgwtik94CYRlrnQ

In [19]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WO

### Time Travel

In [23]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b7e7d-1ca9-652c-8001-43dd22c1d603"}})

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS

In [24]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": '1f1b7e7d-1ca9-652c-8001-43dd22c1d603'}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!',
   'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS75gW88wyk/3ooD

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people wanted to "pizza" (see) him. The joke is a pun that combines the idea of mushrooms being fungi with the concept of being a fun person at a party.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-100a-6bff-8002-7d6c3d37b1f4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:57:21.959833+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-064c-630b-8001-707d60a085ad'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the 

#### Updating State

In [25]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": '1f1b7e7d-1ca9-652c-8001-43dd22c1d603', "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b7e7f-d59a-6430-8002-2706c0c47671'}}

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!', 'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/W

In [28]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": '1f1b7e7d-1ca9-652c-8001-43dd22c1d603'}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza go to therapy?\n\nBecause it had too many **crust** issues!',
   'extras': {'signature': 'Eq8VCqwVAWkUfRNEHbtv1w9w8zWGeCgaEKKF42ntsBb1IDkz6LiEPxqoMxJznHjVQYBqP66u3PSLWGxxl+dHfz63ekJAXfswfl7vkUtfhX567y0LSXYp4bqnQwBzFnXuAZ6MCR/toH9RX6RhQKKC6qGSCwIqqjvZJIAt6EoEWdqLYUQ7QNm4ysNe46F71WQjVzmxV1nCGks3wOq6eYoUmhLgyKEiMga+WReGOLZpTp+AnHOph2MOH010gCyggJUAJDsJj7mstuACly6OhTNo+jYaMxnw9jnjF3wqgMJ7CJaGux1qg8xGQkyd4K+/qiY452Wh6k9u9e7b1i0qdgXtiCHSaae+wC2DAiTl2bKitz2OntY7AShlEbNeDWbMtvwvqPSD1SywttLJIexSDvQaVkw+MbAgjdgxyoLG4zFPEd5E1VocwfGxzfe1Lb/H2rFs/ReHc6oenj7u0nlujnaktWjAgKbjgWcKy5xE52wp5LlAlJO/Qgjwj2Rh30XTMUTFDE+igshWX3pPr73zwo8Nfl2K04cFC9FzLsVGVZteoO1gv7vtNSw+SBGOymEYEpaQ7xefHTOVPHDAUcAicuC05xYCQz1EwUbrJWv5/VoE9POqp0s5Q7IIcdsWD4FZL8RZ+JNzN67dbIr4u9NB6/MtulrnzUCI655wB+iwfieQjkEpw69DlrYYLZKiptTa10dyvkfV/r15urodhdFhS5KcU1w+eouIC1q0Y8vEAYz5U6EY4N0mkvSApdcxCI+HMNn/xLDXn2uX8R19IDF+6yl+mtHmE7qr4+4OleNspbgF2NGkNKNT/WOS75gW88wyk/3ooD

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [29]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))